# CESR planar wiggler: from a four-potential to a transport map

This notebook derives the continuous-field model used for the two periodic planar wigglers in the CESR lattice. It then constructs the corresponding SciBmad element and obtains its linear transport matrix by propagating a GTPSA map through the same symplectic tracking kernel used for particles.

The model is parameterized as

$$
(B_{\max},L_{\mathrm{period}},N_{\mathrm{period}})
\longrightarrow A^\mu(x,y,s,t)
\longrightarrow \mathcal M
\longrightarrow (c,R,T,\ldots).
$$

A Bmad or Tao map is therefore a validation target, not a frozen replacement for the element model.

## 1. Periodic planar-wiggler field

Bmad's periodic planar model can be written as

$$
\begin{aligned}
B_x &=-B_{\max}\frac{k_x}{k_y}\sin(k_xx)\sinh(k_yy)\cos\theta,\\
B_y &= B_{\max}\cos(k_xx)\cosh(k_yy)\cos\theta,\\
B_s &=-B_{\max}\frac{k_s}{k_y}\cos(k_xx)\sinh(k_yy)\sin\theta,
\end{aligned}
$$

with

$$
k_y^2=k_x^2+k_s^2,\qquad k_s=\frac{2\pi}{L_{\mathrm{period}}},\qquad \theta=k_ss+\phi_s.
$$

The CESR file supplies only B_MAX, L_PERIOD, and N_PERIOD, so the corresponding specialization is $k_x=0$ and $k_y=k_s\equiv k_w$:

$$
B_x=0,\qquad B_y=B_{\max}\cosh(k_wy)\cos\theta,\qquad B_s=-B_{\max}\sinh(k_wy)\sin\theta.
$$

The phase $\phi_s=-k_wL/2$ makes the on-axis vertical field symmetric about the element center.

## 2. Constructing the four-potential

For a magnetostatic element, $\phi=0$ and $\mathbf B=\nabla\times\mathbf A$. Gauge freedom allows the particularly simple choice

$$
A^\mu=(\phi,A_x,A_y,A_s),\qquad \phi=A_y=A_s=0,
$$

$$
A_x=\frac{B_{\max}}{k_w}\cosh(k_wy)\sin(k_ws+\phi_s).
$$

Taking the curl gives

$$
B_y=\frac{\partial A_x}{\partial s}=B_{\max}\cosh(k_wy)\cos\theta,
$$

$$
B_s=-\frac{\partial A_x}{\partial y}=-B_{\max}\sinh(k_wy)\sin\theta.
$$

The Julia callback returns the four potential and all sixteen derivatives $\partial A_\alpha/\partial(x,y,s,t)$. Supplying the derivatives analytically avoids nested numerical differentiation inside the implicit Hamiltonian integrator.

## 3. Transport maps and GTPSA

Let $\mathbf z=(x,p_x,y,p_y,z,p_z)^T$ denote canonical phase-space coordinates. Near a reference point $\mathbf z_0$, the tracked map has the expansion

$$
z_i^{\mathrm{out}}=c_i+\sum_jR_{ij}\Delta z_j+\frac12\sum_{jk}T_{ijk}\Delta z_j\Delta z_k+\cdots,
$$

where $\Delta\mathbf z=\mathbf z-\mathbf z_0$ and

$$
c=\mathcal M(\mathbf z_0),\qquad R_{ij}=\left.\frac{\partial\mathcal M_i}{\partial z_j}\right|_{\mathbf z_0}.
$$

GTPSA represents each input coordinate as a truncated polynomial, $z_i=z_{0,i}+\delta_i$. Tracking these six polynomials once propagates every retained coefficient through the element. The Jacobian of the resulting DAMap is therefore $R$ directly; no finite-difference step size is introduced.

For radiation-free Hamiltonian tracking, the first-order matrix should satisfy

$$R^TJR=J,\qquad J=\operatorname{diag}(J_2,J_2,J_2),\qquad J_2=\begin{bmatrix}0&1\\-1&0\end{bmatrix}. $$

In [ ]:
using SciBmad
using GTPSA
using LinearAlgebra
using CairoMakie

cesr_dir = if isfile(joinpath(@__DIR__, "wiggler.jl"))
    @__DIR__
else
    joinpath(@__DIR__, "lattices", "cesr")
end
include(joinpath(cesr_dir, "wiggler.jl"))
using .WigglerModels

In [ ]:
const CESR_B_MAX = 1.17               # T
const CESR_L_PERIOD = 0.19625         # m
const CESR_N_PERIOD = 12
const CESR_L = CESR_N_PERIOD * CESR_L_PERIOD
const CESR_P0C = 5.2889999753148e9    # eV/c

scales = planar_wiggler_scales(
    B_max=CESR_B_MAX,
    L_period=CESR_L_PERIOD,
    p0c=CESR_P0C,
)

The basic scales provide useful checks:

$$B\rho=17.6422\;\mathrm{T\,m},\qquad \rho_w=\frac{B\rho}{B_{\max}}=15.0788\;\mathrm m,$$

$$a_x=\frac{1}{\rho_wk_w^2}=64.698\;\mu\mathrm m,\qquad a_{x'}=\frac{1}{\rho_wk_w}=2.071\;\mathrm{mrad}.$$

The horizontal amplitude reproduces OSC_AMPLITUDE in the source CESR lattice. The leading averaged vertical focusing is

$$\langle K_y\rangle\simeq\frac{1}{2\rho_w^2}=2.1991\times10^{-3}\;\mathrm{m}^{-2}. $$

In [ ]:
k_w = 2pi / CESR_L_PERIOD
phase = -k_w * CESR_L / 2
params = (CESR_B_MAX, k_w, phase)

s_grid = range(0, CESR_L; length=1201)
B_y_axis = [planar_wiggler_field(0.0, 0.0, s, params)[2] for s in s_grid]

fig = Figure(size=(900, 360))
ax = Axis(fig[1, 1], xlabel="local s [m]", ylabel="B_y on axis [T]",
          title="CESR planar-wiggler on-axis field")
lines!(ax, s_grid, B_y_axis, linewidth=2)
fig

In [ ]:
x_test, y_test, s_test, t_test = 1.0e-3, 2.0e-3, 0.37, 0.0
potential, dA = planar_wiggler_four_potential(x_test, y_test, s_test, t_test, params)

# With Ay = As = 0, By = dAx/ds and Bs = -dAx/dy.
B_from_curl = (0.0, dA[7], -dA[6])
B_direct = planar_wiggler_field(x_test, y_test, s_test, params)

(potential=potential, B_from_curl=B_from_curl, B_direct=B_direct,
 max_error=maximum(abs.(B_from_curl .- B_direct)))

## 4. Construct and track the SciBmad element

PlanarWiggler returns a normal Beamlines LineElement with FourPotentialParams and a sixth-order Yoshida integration method. The physical inputs remain the Bmad field and period parameters. With sixteen slices per period, each CESR wiggler uses 192 integration steps.

In [ ]:
wiggler = PlanarWiggler(
    alias="WIG_TEST",
    L=CESR_L,
    B_max=CESR_B_MAX,
    L_period=CESR_L_PERIOD,
    N_period=CESR_N_PERIOD,
    slices_per_period=16,
    order=6,
)

wiggler_line = Beamline([wiggler], species_ref=Species("electron"), E_ref=CESR_P0C)
tracked = track(wiggler_line; v0=zeros(1, 6), use_KA=false, use_explicit_SIMD=false)
zeroth_order_particle = vec(tracked.v[1, :, end])

## 5. Extract the linear map with GTPSA

The helper below creates a first-order Descriptor, forms the six GTPSA variables about the requested reference coordinate, tracks them through the beamline, and returns a DAMap. Calling jacobian on that map reads the transported first-order coefficients.

The small fifth-coordinate constant is physical: a particle oscillating inside the wiggler follows a slightly longer path than the straight reference trajectory.

In [ ]:
wiggler_map = gtpsa_transport_map(wiggler_line; v0=zeros(6), order=1)
R = Matrix(jacobian(wiggler_map))
map_constant = scalar.(wiggler_map.v)

J2 = [0.0 1.0; -1.0 0.0]
J = kron(Matrix{Float64}(I, 3, 3), J2)
symplectic_residual = opnorm(transpose(R) * J * R - J, Inf)

(constant=map_constant, R=R, det_R=det(R),
 symplectic_residual=symplectic_residual)

## 6. Compare with the averaged focusing model

After averaging over the fast oscillation, the horizontal plane is approximately a drift, while the vertical plane is approximately a constant-focusing channel:

$$R_x\simeq\begin{bmatrix}1&L\\0&1\end{bmatrix},$$

$$R_y\simeq\begin{bmatrix}\cos(\sqrt{K_y}L)&\sin(\sqrt{K_y}L)/\sqrt{K_y}\\-\sqrt{K_y}\sin(\sqrt{K_y}L)&\cos(\sqrt{K_y}L)\end{bmatrix}. $$

This approximation is a low-order check only; it does not replace the continuous field.

In [ ]:
K_y = scales.K_y_average
root_K = sqrt(K_y)
R_x_average = [1.0 CESR_L; 0.0 1.0]
R_y_average = [
    cos(root_K * CESR_L)           sin(root_K * CESR_L) / root_K
   -root_K * sin(root_K * CESR_L)  cos(root_K * CESR_L)
]

(horizontal_gtpsa=R[1:2, 1:2], horizontal_average=R_x_average,
 vertical_gtpsa=R[3:4, 3:4], vertical_average=R_y_average)

## 7. Bmad validation and the current full-ring Twiss status

For a Bmad comparison, inspect WIG_W and WIG_E with Tao and compare the constant vector, all entries of the 6 by 6 matrix, path length, off-momentum terms, and the full-ring tunes. Continuous-field Bmad tracking is the closest reference for this four-potential model; bmad_standard uses a specialized planar-wiggler approximation and need not agree term by term.

Three independent issues were separated while diagnosing the current full CESR model:

1. The RF cavities are disabled, so CESR must be treated as a coasting ring. Finite-difference coast detection produces a tiny numerical derivative and can misclassify it as bunched; set coasting_beam=true when diagnosing the closed orbit.
2. BeamTracking's implicit four-potential kernel allocates temporary dynamic TPS objects from GTPSA.desc_current. SciBmad restores the previous global descriptor when it creates a default temporary descriptor, which produces No Descriptor defined. Create a Descriptor explicitly and pass it as GTPSA_descriptor until the upstream allocation is fixed.
3. A four-dimensional coasting closed-orbit Newton solve driven entirely by GTPSA converges in three iterations to a residual below 5e-14. Normal-form analysis then reaches the physical stability check and reports an unstable orbital eigenmode. The no-kicker one-turn GTPSA matrix has transverse eigenvalue magnitudes approximately (10.0540, 1, 1, 0.09946). Replacing both wigglers by drifts still gives approximately (9.2743, 1, 1, 0.10782), so the wiggler model is not the cause of the Twiss failure. The remaining task is an element-by-element Bmad/SciBmad map comparison of the converted baseline, especially combined superimposed IR elements and bend/fringe conventions.